In [2]:
!pip install unsloth -q
!pip install "transformers>=4.50.3" -q
!pip install trl==0.15.2 -q
print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 30.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.2 MB/s eta 0:00:00


In [8]:
from huggingface_hub import login
login(token="hf_YOUR_TOKEN_HERE")
print("Logged in!")

Logged in!


In [9]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    max_seq_length = 512,
    load_in_4bit = True,
)
print("Model loaded!")

==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Model loaded!


In [10]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA ready!")

LoRA ready!


In [12]:
import json
import random
from datasets import Dataset

with open('/kaggle/input/datasets/elvis23/mental-health-conversational-data/intents.json', 'r') as f:
    data = json.load(f)

kaggle_data = []
for intent in data['intents']:
    tag = intent['tag']
    patterns = intent['patterns']
    responses = intent['responses']
    if tag in ['greeting', 'goodbye', 'thanks']:
        continue
    for pattern in patterns:
        if len(pattern) > 10:
            response = random.choice(responses)
            kaggle_data.append({"instruction": pattern, "response": response})

indian_data = [
    {"instruction": "I failed my NEET exam and I can't face my parents.", "response": "The weight of feeling like you've let down the people who gave everything for you — that's one of the heaviest things a person can carry. What are you most afraid they will say?"},
    {"instruction": "My parents are forcing me into an arranged marriage and I don't want it.", "response": "Being caught between your own heart and the people you love most — that pressure is exhausting in a way that's hard to explain to anyone outside it. What does the thought of saying something feel like for you?"},
    {"instruction": "நான் மிகவும் தனிமையாக உணர்கிறேன். என் குடும்பம் என்னை புரிந்துகொள்வதில்லை.", "response": "உங்கள் சொந்த குடும்பத்தினரால் புரிந்துகொள்ளப்படாத உணர்வு மிகவும் வலிக்கும். அந்த தனிமை உங்களை எப்போதிருந்து பாதிக்கிறது?"},
    {"instruction": "I lost my job and I'm ashamed to tell my family.", "response": "Carrying that alone — the fear of disappointing the people who depend on you — is an enormous burden. How long have you been holding this by yourself?"},
    {"instruction": "मुझे हर रात रोना आता है और मैं नहीं जानता क्यों।", "response": "बिना किसी कारण के रोना बहुत थका देने वाला होता है। यह कितने समय से हो रहा है आपके साथ?"},
    {"instruction": "I scored low in my board exams and my family is disappointed.", "response": "Feeling worthless after one result — that pain makes complete sense. What's going through your mind when you're alone?"},
    {"instruction": "I don't want to be here anymore. Everything feels pointless.", "response": "I hear you, and I'm really glad you said that out loud. Please reach out to iCall right now at 9152987821 — they will listen without judgment. Can you tell me what's been the hardest part of these days?"},
    {"instruction": "I have anxiety before every exam. My hands shake and I can't think.", "response": "That feeling of your body going into panic — the shaking hands, the mind going blank — does it start the night before, or even earlier?"},
    {"instruction": "I'm a software engineer but I hate my job. My parents think I'm successful but I'm dying inside.", "response": "Living a life that looks successful from the outside while feeling like you're disappearing inside — that gap is exhausting to maintain every day. What does a typical day feel like for you right now?"},
    {"instruction": "I think I have depression but my family doesn't take mental health seriously.", "response": "Recognizing something is wrong inside you, but knowing your family won't understand — that isolation makes the pain even heavier. What made you start feeling this way, do you think?"},
    {"instruction": "I am always compared to my cousin who is successful and I feel I am never enough.", "response": "Being measured against someone else your whole life — never being seen for who you actually are — slowly chips away at a person. When did these comparisons start for you?"},
    {"instruction": "I can't sleep. My mind won't stop thinking.", "response": "That racing mind at night when everything is quiet except your own thoughts — what tends to be the thought that starts it all?"},
    {"instruction": "என் தந்தை என்னை எப்போதும் திட்டுவார். வீட்டிற்கு வர பயமாக இருக்கிறது.", "response": "உங்கள் சொந்த வீடே பயமாக இருக்கும்போது அதுவே மிகவும் கஷ்டமான நிலை. இப்போது நீங்கள் பாதுகாப்பாக இருக்கிறீர்களா?"},
    {"instruction": "मेरे दोस्त मुझसे दूर हो गए हैं और मैं बहुत अकेला महसूस कर रहा हूं।", "response": "दोस्तों का दूर हो जाना बहुत गहरी तकलीफ देता है। यह अकेलापन कब से महसूस हो रहा है आपको?"},
    {"instruction": "My husband doesn't understand my feelings and I feel very alone in this marriage.", "response": "Feeling alone inside a marriage where you expected partnership — has there been a moment recently that made this feeling especially sharp?"},
]

all_data = kaggle_data + indian_data

def format_conversation(example):
    return {"text": f"""<start_of_turn>user
{example['instruction']}<end_of_turn>
<start_of_turn>model
{example['response']}<end_of_turn>"""}

dataset = Dataset.from_list(all_data)
dataset = dataset.map(format_conversation)
print(f"Dataset ready: {len(all_data)} examples!")

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Dataset ready: 192 examples!


In [13]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 512,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = False,
        bf16 = False,
        logging_steps = 1,
        output_dir = "/kaggle/working/mentalbridge_output",
        optim = "adamw_8bit",
        save_steps = 20,
        save_total_limit = 1,
        seed = 42,
        report_to = "none",
        dataset_num_proc = 2,
    ),
)
print("Starting training...")
trainer.train()
print("Training complete!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/192 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 192 | Num Epochs = 3 | Total steps = 72
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Step,Training Loss
1,7.551783
2,7.430250
3,7.350236
4,6.880056
5,7.262586
6,7.818313
7,7.429509
8,6.162338
9,6.087196
10,5.285000


Training complete!


In [18]:
import shutil
import os

# Remove the failed GGUF conversion files
if os.path.exists('/kaggle/working/mentalbridge_gemma4_gguf'):
    shutil.rmtree('/kaggle/working/mentalbridge_gemma4_gguf')
    print("Removed gguf folder")

if os.path.exists('/kaggle/working/mentalbridge_gemma4_gguf_gguf'):
    shutil.rmtree('/kaggle/working/mentalbridge_gemma4_gguf_gguf')
    print("Removed gguf_gguf folder")

if os.path.exists('/kaggle/working/mentalbridge_output'):
    shutil.rmtree('/kaggle/working/mentalbridge_output')
    print("Removed output folder")

# Check remaining space
import subprocess
result = subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True)
print(result.stdout)

Removed gguf folder
Removed gguf_gguf folder
Removed output folder
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.2M   20G   1% /kaggle/working



In [21]:
from huggingface_hub import HfApi
api = HfApi()

api.create_repo(
    repo_id="vishnuprasadgs/mentalbridge-gemma4-E2B-finetuned",
    private=False,
    exist_ok=True
)

# Push just the LoRA adapter — very small size
model.push_to_hub(
    "vishnuprasadgs/mentalbridge-gemma4-E2B-finetuned",
    token="hf_YOUR_TOKEN_HERE"
)
tokenizer.push_to_hub(
    "vishnuprasadgs/mentalbridge-gemma4-E2B-finetuned",
    token="hf_YOUR_TOKEN_HERE"
)
print("Pushed to HuggingFace!")

Saved model to https://huggingface.co/vishnuprasadgs/mentalbridge-gemma4-E2B-finetuned
Pushed to HuggingFace!
